# W09｜Reliability / Retry / Idempotency / HITL｜教学版

**建议学习时间：5h**

这本 Notebook 的定位是“第二教材”，不是代码仓库。  
正确使用方式：

1. 先看课程视频；
2. 回到 Notebook 看“是什么 / 为什么 / 什么时候用”；
3. 带着“你应该观察什么”再运行代码；
4. 先自己做题，再展开参考答案；
5. 最后把同一机制接入 ChatBI。

## Excel 任务表 1:1 对照

| # | 模块 | Excel任务 | 输出/验收 | Notebook处理方式 |
|---:|---|---|---|---|
| 1 | Backend | 按下方课程映射完成错误处理 + Reliability | 重点是 retry/idempotency/failure | 知识点 + Checkpoint/参考答案 |
| 2 | Agent | 学 HITL / Tool Failure Fallback | 知道高风险动作何时需确认 | 知识点 + Checkpoint/参考答案 |
| 3 | Lab | 重复 POST + Tool 500 故障实验 | 观察重复副作用 | 明确作业 + 参考答案/参考实现 |
| 4 | ChatBI | 加入 idempotency + 基础 retry | 重复请求只生成一个 Job | 明确作业 + 参考答案/参考实现 |
| 5 | 复盘 | 写一个 Failure Flow | 能解释失败后怎么安全重试 | 明确作业 + 参考答案/参考实现 |

> **规则**：Task 1-2 是学习任务，用 Checkpoint/面试题验收；Task 3-5 是作业任务，下面给明确参考答案/参考实现。设计题没有唯一答案时给参考图/评分标准。

## Excel 作业参考答案 / 验收标准

### Task 3｜Duplicate POST + Tool 500
必须分别观察：
- 没幂等时 duplicate POST 是否创建 2 个 job；
- transient Tool 500/503 是否触发 limited retry；
- permanent validation/permission error 不应 retry。

### Task 4｜验收
同一个 `Idempotency-Key` 重复请求只能创建一个 Job；Retry 使用 backoff 且有次数上限。

### Task 5｜Failure Flow
`Validate → Idempotency Check → Tool Call → classify error → transient: retry/backoff | permanent: fail`

> 建议：先完成作业，再对照参考答案。不要直接照抄。

## 本周学习目标

        - 建立可靠性基础
- 会区分 transient/permanent error
- 设计幂等请求
- 理解 HITL

        ## 本周前置检查

        - 完成 W08；理解 long job / queue

        ## 本周必须记住的英文术语

        - `retry`：重试
- `exponential backoff`：指数退避
- `idempotency`：幂等性
- `circuit breaker`：熔断器
- `HITL`：人在环路

        > 可以听中文翻译，但这些英文术语一定要保留，因为代码、日志、文档和面试都会使用英文。

## 1. Timeout ≠ Failure

    **是什么（What）**  
    客户端超时只说明在规定时间内没收到结果，不代表服务端没有执行成功。

    **为什么需要（Why）**  
    如果你简单 retry，有副作用的操作可能执行两次。

    **什么时候用（When）**  
    创建任务、调用支付/发送/外部审批等。

    **常见误区 / 容易踩坑**  
    不要把 timeout 当作“什么都没发生”。


**中英术语**：`timeout` = 超时；`retry` = 重试；`idempotency` = 幂等；`backoff` = 退避

### 代码实验：重试一个临时 Timeout

**这段代码想让你看什么？**

观察只对 TimeoutError 重试，并使用指数退避。

> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
import time

def retry(fn, attempts=4, base_delay=0.02):
    last = None
    for attempt in range(attempts):
        try:
            return fn()
        except TimeoutError as e:
            last = e
            if attempt < attempts - 1:
                time.sleep(base_delay * (2 ** attempt))
    raise last

counter = {"n": 0}
def flaky():
    counter["n"] += 1
    if counter["n"] < 3:
        raise TimeoutError("temporary")
    return "OK"

print(retry(flaky), counter)

## 2. Idempotency

    **是什么（What）**  
    相同业务请求重复执行时，业务效果与执行一次相同。

    **为什么需要（Why）**  
    网络 retry、queue redelivery、用户双击都会产生重复请求。

    **什么时候用（When）**  
    POST create job、外部 side-effect tool。

    **常见误区 / 容易踩坑**  
    只在应用内存 dict 里存 key 不够生产级；通常要数据库 unique constraint / durable record。


**中英术语**：`idempotency key` = 幂等键；`duplicate request` = 重复请求；`unique constraint` = 唯一约束

In [ ]:
store = {}

def create_job(idempotency_key, payload):
    if idempotency_key in store:
        return store[idempotency_key], False
    job = {"id": len(store)+1, "payload": payload}
    store[idempotency_key] = job
    return job, True

print(create_job("same-key", {"q":"trend"}))
print(create_job("same-key", {"q":"trend"}))

## 3. HITL（Human-in-the-loop）

    **是什么（What）**  
    在高风险动作执行前暂停，等待人类确认。

    **为什么需要（Why）**  
    模型可能误解意图；高风险副作用不能只靠概率模型自动决定。

    **什么时候用（When）**  
    导出敏感数据、发送消息、删除数据、执行外部动作。

    **常见误区 / 容易踩坑**  
    HITL 不是每一步都弹确认；应该根据风险分层。


**中英术语**：`human-in-the-loop` = 人在环路；`interrupt` = 中断等待；`resume` = 恢复

### 练习：错误是否应该重试？

判断：LLM 429、Tool 503、权限不足、SQL validator rejected、DB deadlock。

<details>
    <summary><b>参考答案</b></summary>

    ```python
# LLM 429 -> 通常可重试，带 Retry-After/backoff
# Tool 503 -> 通常可重试
# 权限不足 -> 不重试
# SQL validator rejected -> 不重试，需改 plan/input
# DB deadlock -> 某些场景可在事务层有限次数重试
```

    </details>

### Checkpoint

1. 为什么 Retry 一定要与 Idempotency 一起考虑？
2. 什么场景适合 HITL？
3. 为什么无限 retry 很危险？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** Retry 可能重复副作用；幂等保证重复执行不会产生第二份业务效果。

**2.** 高风险、不可逆、有外部副作用或权限敏感的 Agent 动作。

**3.** 会放大下游故障、形成重试风暴，还可能增加延迟和成本。

    </details>

## ChatBI 本周作业

        给 `POST /analysis` 增加 `Idempotency-Key`；  
对一个高风险 Tool 增加 `PENDING_APPROVAL → APPROVED/BLOCKED` 状态。

### 本周面试 / 理解检查

1. 为什么 timeout 后不能直接假设失败？
2. 幂等键通常放哪里保存？
3. 什么错误不应该 retry？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** 远端可能已经执行完成，只是响应没有及时返回。

**2.** 生产上应放持久化数据库/幂等记录表，并用唯一约束保证并发安全。

**3.** 权限、业务校验、参数/schema、非法 plan 等确定性错误。

    </details>

## 本周结束前的自检

只有下面 4 项都能做到，才算真正完成：

- [ ] 我能不用课程原话，用自己的话解释核心概念；
- [ ] 我知道这周的代码实验到底在证明什么；
- [ ] 我至少独立完成了一道代码题；
- [ ] 我能说出一个“生产环境里如果做错会发生什么”的例子。